# D — V5 stage 1: mT5-small dialogue training on public pairs

Run on a Kaggle **GPU T4** notebook with Internet on. It trains `google/mt5-small` locally on DailyDialog (tagged `STANDARD`) and CoSEM (tagged `LOCAL`) reply pairs. No hosted LLM API calls, and no ReplyCoach development or test item is read.

Before running:

1. Push the branch that contains `src/replycoach/generate/train_v5.py`.
2. Add a Kaggle secret named `GITHUB_TOKEN` (read-only access to the private repo) under Add-ons → Secrets.
3. Upload the four pair files as a **private** Kaggle dataset and attach it: `dailydialog_train_pairs.jsonl`, `dailydialog_validation_pairs.jsonl` (Role C's files, so V4 and V5 see the same DailyDialog), `cosem_train_pairs.jsonl`, `cosem_validation_pairs.jsonl` (from `scripts/prepare_cosem_pairs.py`). Both corpora are CC BY-NC-SA 4.0; do not make the dataset public.

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient

BRANCH = "main"
REPO = "/kaggle/temp/ReplyCoachSG"  # outside /kaggle/working, so the clone is never saved as notebook output
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
result = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH,
     f"https://{token}@github.com/jerememetan/ReplyCoachSG.git", REPO],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise SystemExit("clone failed: check the GITHUB_TOKEN secret, the branch name, and repo access")
subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/jerememetan/ReplyCoachSG.git"], check=True)
!pip -q install --no-deps -e {REPO}
!pip -q install sentencepiece

In [ ]:
from pathlib import Path
import torch

DATA = Path("/kaggle/input/replycoach-public-pairs")  # change to your dataset's folder
OUT = Path("/kaggle/working/artifacts")
TRAIN = [DATA / "dailydialog_train_pairs.jsonl", DATA / "cosem_train_pairs.jsonl"]
VALIDATION = [DATA / "dailydialog_validation_pairs.jsonl", DATA / "cosem_validation_pairs.jsonl"]
missing = [str(path) for path in TRAIN + VALIDATION if not path.exists()]
assert not missing, f"attach the pairs dataset; missing: {missing}"
assert torch.cuda.is_available(), "enable the GPU: Settings -> Accelerator -> GPU T4"
train_args = " ".join(map(str, TRAIN))
validation_args = " ".join(map(str, VALIDATION))
print(torch.cuda.get_device_name(0))

## Smoke run

About 4,000 pairs for one epoch. Check that the loss falls, that no step reports a non-finite loss, and how many seconds 200 steps take before starting the full run.

In [ ]:
!python -m replycoach.generate.train_v5 --train-pairs {train_args} --validation-pairs {validation_args} \
    --train-limit 2000 --validation-limit 200 --epochs 1 --eval-every 50 --warmup-steps 20 \
    --output-name mt5_v5_smoke --output-dir /kaggle/temp/smoke

## Full run

Validation and a resumable checkpoint are written every 2,000 steps. If the session stops, save a version, attach its output as an input, and add `--resume-from <that input>/artifacts/mt5_v5_pretrain/training_state.pt` to the same command.

In [ ]:
!python -m replycoach.generate.train_v5 --train-pairs {train_args} --validation-pairs {validation_args} \
    --epochs 3 --output-name mt5_v5_pretrain --output-dir {OUT}

In [ ]:
import json

report = json.loads((OUT / "mt5_v5_pretrain.json").read_text(encoding="utf-8"))
print({key: report[key] for key in ("parameter_count", "device_name", "peak_cuda_gb", "best_step", "best_validation_loss")})
for entry in report["history"]:
    print(entry)
for example in report["examples"]:
    print(json.dumps(example, ensure_ascii=False, indent=2))

## Optional ablation: DailyDialog only

Same settings without CoSEM. This is the V5 that matches what V4 was trained on, so the report can separate the effect of the architecture from the effect of the Singapore chat data. Run it in a separate session if disk space is short: each run keeps about 5 GB.

In [ ]:
# !python -m replycoach.generate.train_v5 --train-pairs {TRAIN[0]} --validation-pairs {VALIDATION[0]} \
#     --epochs 3 --output-name mt5_v5_pretrain_dd --output-dir {OUT}

## After the run

- Save a version so `artifacts/mt5_v5_pretrain/best/` is kept; the fine-tune starts from that folder.
- Copy `mt5_v5_pretrain.json` into the team repo's `experiments/` notes. The example replies contain CoSEM text: check them for unscrubbed personal details before pasting any into the report.